In [2]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py

--2026-07-16 07:48:45--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.


HTTP request sent, awaiting response... 200 OK
Length: 1376 (1.3K) [text/plain]
Saving to: ‘download.py’

download.py         100%[===================>]   1.34K  --.-KB/s    in 0s      

2026-07-16 07:48:45 (55.2 MB/s) - ‘download.py’ saved [1376/1376]

--2026-07-16 07:48:45--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1520 (1.5K) [text/plain]
Saving to: ‘embedder.py’

embedder.py         100%[===================>]   1.48K  --.-KB/s    in 0.001s  

2026-07-16 07:48:46 (2.59 MB/s) - ‘embedder.py’ saved [1520/1520]



In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

# Q1. Embedding a query
Embed the following query:

How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

- -0.31
- -0.02
- 0.12
- 0.44

In [3]:
from embedder import Embedder
embedder = Embedder()

query = "How does approximate nearest neighbor search work?"
query_vector = embedder.encode(query)
print(query_vector.shape)
print(query_vector[0])

(384,)
-0.02058203437252893


# Q2. Cosine similarity
The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1. What do you get?

- 0.07
- 0.37
- 0.68
- 0.92

In [5]:
import numpy as np

target = next(doc for doc in documents if doc["filename"].endswith("07-sqlitesearch-vector.md"))

doc_vec = embedder.encode(target["content"])

cos_sim = float(np.dot(query_vector, doc_vec))
print(cos_sim)

0.36107027225589694


# Q3. Chunking and search by hand
A full page covers several topics, which waters down its embedding.
We chunk the pages the same way as in homework 1

Q: Which file does the highest-scoring chunk belong to (its filename)?

- 02-vector-search/lessons/03-embeddings-dataset.md
- 02-vector-search/lessons/06-rag-vector.md
- 02-vector-search/lessons/07-sqlitesearch-vector.md
- 02-vector-search/lessons/09-onnx-embedder.md


In [6]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

chunk_vectors = [embedder.encode(chunk["content"]) for chunk in chunks]
scores = [float(np.dot(query_vector, chunk_vector)) for chunk_vector in chunk_vectors]

top_idx = int(np.argmax(scores))
best_chunk = chunks[top_idx]
best_score = scores[top_idx]

print(best_score)
print(best_chunk["filename"])
print(best_chunk["content"][:500])

0.6489017718578812
02-vector-search/lessons/07-sqlitesearch-vector.md
rch. We score
the query against every document and pick the top ones. It always finds
the true top matches, but it pays for that by touching everything.

Approximate nearest neighbor (ANN) search takes a shortcut. Instead of
comparing against everything, it first narrows down to a region of
likely matches. Then it scores only within that region. It may miss the
absolute best match, but the results are still good and it's much
faster.

```text
NN (exact):    compare query against ALL documents ->


# Q4. Vector search with minsearch
We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

What metric do we use to evaluate a search engine?

Q: Which file is the filename of the first result?

- 02-vector-search/lessons/04-vector-search.md
- 04-evaluation/lessons/05-search-metrics.md
- 04-evaluation/lessons/13-llm-as-judge.md
- 05-monitoring/lessons/04-metrics.md

In [ ]:
from minsearch import VectorSearch
vsearch = VectorSearch()

document_vectors = [embedder.encode(doc["content"]) for doc in documents]
document_vectors = np.array(document_vectors)
vsearch.fit(document_vectors, documents)

q4 = "What metric do we use to evaluate a search engine?"
q4_vector = embedder.encode(q4)

results = vsearch.search(q4_vector)
# print(results)

results[0]["filename"]

[{'content': '# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup, each query

'04-evaluation/lessons/05-search-metrics.md'

# Q5. Text search vs vector search
Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

- 02-vector-search/lessons/01-intro.md
- 02-vector-search/lessons/02-embeddings.md
- 02-vector-search/lessons/08-pgvector.md
- 03-orchestration/lessons/05-rag.md

In [20]:
q5 = "How do I store vectors in PostgreSQL?"

vector_results = vsearch.search(embedder.encode(q5))
vector_files = [r["filename"] for r in vector_results[:5]]
print(vector_files)





['02-vector-search/lessons/08-pgvector.md', '05-monitoring/lessons/05-database.md', '02-vector-search/lessons/02-embeddings.md', '02-vector-search/lessons/04-vector-search.md', '02-vector-search/lessons/07-sqlitesearch-vector.md']


In [21]:
from minsearch import Index

index = Index(
    text_fields=["content"]
)

index.fit(documents)

q5 = "How do I store vectors in PostgreSQL?"
text_results = index.search(q5)
text_files = [r["filename"] for r in text_results[:5]]
print(text_files)

['02-vector-search/lessons/02-embeddings.md', '02-vector-search/lessons/01-intro.md', '02-vector-search/lessons/03-embeddings-dataset.md', '03-orchestration/lessons/05-rag.md', '02-vector-search/lessons/10-next-steps.md']


# Q6. Hybrid search
Both vector and text search have their strengths and weaknesses. Vector search matches by meaning, so it finds relevant pages even when they use words different from the query. But it can miss exact terms like names, codes, or rare keywords. Text search is the opposite: it nails exact words but misses paraphrases and synonyms.

We don't have to pick one or the other - we can use both and merge their results. This approach is called "hybrid search".

Each search produces its own ranked list, so we need a way to combine them into one. In this homework we use Reciprocal Rank Fusion (RRF). It ignores the raw scores from each method, which live on different scales and aren't directly comparable. Instead, it looks only at the position of each document in each list.

Every document scores by its position (rank, starting at 0) in each list, and we sum the scores across lists with a constant k = 60:

RRF(d) = sum over lists of  1 / (k + rank(d))

"Sum over lists" means we go through every ranked list and, for each list where the document appears, add its 1 / (k + rank) contribution. A document found by both searches collects a score from each list, while one found by only a single search collects just one.

The constant k controls how much the exact rank matters. A larger k flattens the gap between positions, so the difference between rank 0 and rank 5 counts for less. A smaller k does the opposite: it sharpens that gap, so being at the top of a list matters much more.

The value 60 comes from the original RRF paper and is the usual default. You rarely need to tune it. Lower it when only the top results matter. Raise it to reward documents that appear across many lists, even when they never quite reach the top.

In [24]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc.get("start", 0))
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

# Q6 
Which file is ranked first after RRF?

- 01-agentic-rag/lessons/01-intro.md
- 01-agentic-rag/lessons/13-function-calling.md
- 01-agentic-rag/lessons/14-agentic-loop.md
- 01-agentic-rag/lessons/16-other-frameworks.md




In [ ]:
q6 = "How do I give the model access to tools?"

vector_results = vsearch.search(embedder.encode(q6))

text_results = index.search(q6)

results = rrf([vector_results, text_results])
results

[{'content': "# Other Frameworks\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=4yiCbKX9RhI&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nThe concepts you learned in Part 2 are the same across frameworks.\nFunction calling, the agent loop, and tool definitions all wrap the\nsame pattern. Send messages, run any function calls, and repeat until\nthe model is done.\n\nYou now understand how the loop works. So you can pick up any\nproduction framework and know what it's doing under the hood. I kept\nthis module framework agnostic on purpose, so you can explore and pick\nthe one you like.\n\nHere are some frameworks worth a look:\n\n## OpenAI Agents SDK\n\nThe official SDK from OpenAI for building agents. It uses the same\nResponses API we used throughout this module. It supports tool\ndefinition, multi-turn conversations, and handoffs between agents.\n\n```bash\nuv add openai-agents\n```\n\nGood choice if you're already using OpenAI and want something\nofficial and well-mainta